# AG2 101 (AutoGen): Complete Tutorial

Estimated time needed: **45** minutes

This notebook provides a comprehensive guide to AG2 (formerly AutoGen) basic concepts with runnable examples and detailed explanations. AG2 is an open-source programming framework for building AI agents and facilitating cooperation among multiple agents to solve tasks.

## Table of Contents

<ol>
    <li><a href="#Setup-and-Import">Setup and Import</a></li>
    <li><a href="#Configure-API-Keys">Configure API Keys</a></li>
    <li>
        <a href="#Introduction-to-Agent-Concepts">Introduction to Agent Concepts</a>
        <ol>
            <li><a href="#Conversable-Agent">Conversable Agent</a></li>
            <li><a href="#Creating-Specialized-Agents">Creating Specialized Agents</a></li>
            <li><a href="#Built-in-Agent-Types">Built-in Agent Types</a></li>
        </ol>
    </li>
    <li><a href="#Human-in-the-Loop">Human-in-the-Loop</a></li>
    <li>
        <a href="#Agent-Orchestration-&-Multi-Agent-Systems-in-AG2">Agent Orchestration & Multi-Agent Systems in AG2</a>
        <ol>
            <li><a href="#GroupChat-and-GroupChatManager">GroupChat and GroupChatManager</a></li>
        </ol>
    </li>
    <li><a href="#Tools-and-Extensions">Tools and Extensions</a></li>
    <li><a href="#Structured-Outputs">Structured Outputs</a></li>
    <li><a href="#Best-Practices">Best Practices</a></li>
    <li><a href="#Conclusion">Conclusion</a></li>
    <li><a href="#Authors">Authors</a></li>
    
</ol>


In [1]:
# AG2 v1+ is a new framework (import ag2). This notebook uses Classic AG2 (import autogen).
%pip install "autogen[openai]" python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import necessary modules
import os
from dotenv import load_dotenv
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from autogen.llm_config import LLMConfig
import json
import time
import random

# Load environment variables
load_dotenv()

print("AG2 modules imported successfully!")

AG2 modules imported successfully!


In [3]:
import logging

# Suppress API key format warning
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

### Configure API Keys

Autogen supports various LLM providers. Let's set up configuration for Ollama.  


In [4]:
%pip import ollama autogen-ext[ollama]

Note: you may need to restart the kernel to use updated packages.


ERROR: unknown command "import"



## Introduction to Agent Concepts

We have several agent concepts in AG2 to help you build your AI agents. We introduce the most common ones here:

- **Conversable agent**: Agents that are able to send messages, receive messages and generate replies using GenAI models, non-GenAI tools, or human inputs.

- **Human in the loop**: Add human input to the conversation.

- **Orchestrating multiple agents**: Users can orchestrate multiple agents with built-in conversation patterns such as swarms, group chats, nested chats, sequential chats or customize the orchestration by registering custom reply methods.

- **Tools**: Programs that can be registered, invoked and executed by agents.

- **Advanced concepts**: AG2 supports more concepts such as structured outputs, RAG, code execution, and so on.


### Conversable Agent

The ConversableAgent is the fundamental building block of AG2, designed to enable seamless communication between AI entities. This core agent type handles message exchange and response generation, serving as the base class for all agents in the framework.

#### Key characteristics:
- **Communication**: Can send and receive messages
- **Processing**: Handles information and generates responses
- **Personality**: Defined by system messages
- **Flexibility**: Base for all other agent types

In the example below, we'll create a simple information validation workflow with two specialized agents that communicate with each other.

**Note**: `initiate_chat()` starts a conversation with another agent.


In [6]:
import autogen

# **Conversable agent**: Agents that are able to send messages, receive messages and generate replies using GenAI models, non-GenAI tools, or human inputs.

# **Human in the loop**: Add human input to the conversation.

llm_config = [
    {
        "model": "gemma4",
        "base_url": "http://localhost:11434/v1",  # Crucial: include the /v1 suffix
        "api_key": "ollama",                     # Required placeholder text
    }
]



# Student agent
student = ConversableAgent(
    name="student",
    system_message="You are a curious student. You ask clear, specific questions to learn new concepts.",
    human_input_mode="NEVER",
    llm_config=llm_config
)

# Tutor agent
tutor = ConversableAgent(
    name="tutor",
    system_message="You are a helpful tutor who provides clear and concise explanations suitable for a beginner.",
    human_input_mode="NEVER",
    llm_config=llm_config
)

# Start conversation
chat_result = student.initiate_chat(
    recipient=tutor,
    message="Can you explain what a neural network is?",
    max_turns=2,
    summary_method="reflection_with_llm"
)

print("\nFinal Summary:")
print(chat_result.summary)

student (to tutor):

Can you explain what a neural network is?

--------------------------------------------------------------------------------
tutor (to student):

That is a huge and fascinating topic! Don't worry, we can break down what a neural network is using analogies.

Simply put, **a neural network is a computer programming model designed to mimic the way the human brain learns and solves problems.**

Instead of being explicitly told the rules (like traditional software), a neural network learns the rules by looking at massive amounts of examples.

---

### 🧠 1. The Basic Concept (The Analogy)

Think of a neural network like a group of highly specialized students that are being taught to recognize objects, like telling the difference between a picture of a cat and a picture of a dog.

1. **You show them examples:** You give them thousands of labeled pictures.
2. **They figure out the patterns:** They slowly learn the patterns that define "cat-ness" or "dog-ness" by adjusting t

#### Why Add summary_method=`"reflection_with_llm"`?

By enabling summary_method=`"reflection_with_llm"`, we allow the system to generate a final response that is not simply a direct answer, but a **reflection of a structured conversation** between multiple agents with defined roles.

In this case, instead of prompting an LLM directly with *“What is a neural network?”*, the system simulates a conversation where:
- A **student** asks a clear question,
- A **tutor** responds with a beginner-friendly explanation,
- And the LLM reflects on this interaction to generate a detailed and coherent summary.
  
This approach can also be extended to **solve complex tasks** by simulating a structured conversation between **expert agents** (for example, a data scientist, a software engineer, and a business strategist), each contributing from their expertise to build a better solution.

This often produces **higher-quality, context-rich answers**, as the reasoning is distributed and clarified through role-based collaboration.  

**Note:** You can try running the conversation without adding the line `summary_method="reflection_with_llm"` and see the difference. 


## Creating Specialized Agents

The power of ConversableAgent comes from customization. Let's create agents with different personalities and purposes:


In [7]:
# Create a Technical Expert Agent
tech_expert = ConversableAgent(
    name="tech_expert",
    system_message="""You are a senior software engineer with expertise in Python, AI, and system design.
    Provide technical, detailed explanations with code examples when appropriate.
    Always consider best practices and performance implications.""",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

# Create a Creative Writer Agent
creative_writer = ConversableAgent(
    name="creative_writer",
    system_message="""You are a creative writer and storyteller.
    Your responses are engaging, imaginative, and use vivid descriptions.
    You excel at making complex topics accessible through stories and analogies.""",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

# Create a Business Analyst Agent
business_analyst = ConversableAgent(
    name="business_analyst",
    system_message="""You are a business analyst focused on ROI, efficiency, and strategic planning.
    Always consider business impact, costs, and practical implementation.
    Provide actionable recommendations with clear metrics.""",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

agents = [tech_expert, creative_writer, business_analyst]
print("Specialized agents created!")
for agent in agents:
    print(f"- {agent.name}: {agent.system_message.split('.')[0]}.")

Specialized agents created!
- tech_expert: You are a senior software engineer with expertise in Python, AI, and system design.
- creative_writer: You are a creative writer and storyteller.
- business_analyst: You are a business analyst focused on ROI, efficiency, and strategic planning.


## Built-in Agent Types

AG2 provides specialized agent classes built on `ConversableAgent` to streamline common workflows such as task-solving, tool use, and user interaction.

#### AssistantAgent — Task-solving LLM assistant

`AssistantAgent` is a subclass of `ConversableAgent` configured with a default system message tailored for solving tasks using LLMs. It can suggest Python code blocks, offer debugging suggestions, and provide structured responses.

- `human_input_mode`: Defaults to `"NEVER"` — the assistant operates autonomously.
- `code_execution_config`: Defaults to `False` — it does **not execute code** itself.
- Designed to work collaboratively with other agents (for example, `UserProxyAgent`) that handle execution.

This agent excels at reasoning, planning, and generating code — and expects others to handle the execution layer.

#### UserProxyAgent — Executing code on behalf of the user

`UserProxyAgent` is a subclass of `ConversableAgent` that acts as a proxy for the human user. It is designed to **execute code**, simulate user decisions, and provide execution-based feedback to other agents like `AssistantAgent`.

- `human_input_mode`: Defaults to `"ALWAYS"` — prompts the user at every turn.
- `llm_config`: Defaults to `False` — no LLM responses unless explicitly configured.
- **Code execution is enabled by default.**

You can customize its behavior by:
- Registering an auto-reply function via `.register_reply()`.
- Overriding `.get_human_input()` to change how user input is gathered.
- Overriding `.execute_code_blocks()`, `.run_code()`, or `.execute_function()` to control code execution behavior.

These two agents are often paired: the `AssistantAgent` writes code, and the `UserProxyAgent` executes it.


In [9]:
%pip install matplotlib numpy pyautogen

  Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---------------------------------------- 9.3/9.3 MB 58.3 MB/s eta 0:00:00
Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 71.7 MB/s eta 0:00:00
Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl (7.2 MB)
Using cached jsonref-1.1.0-py3-none-any.whl (9.4 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Requirements:
# !pip install matplotlib numpy pyautogen

from autogen import AssistantAgent, UserProxyAgent
from autogen.coding import LocalCommandLineCodeExecutor

# Step 1: LLM configuration
llm_config = [
    {
        "model": "gemma4",
        "base_url": "http://localhost:11434/v1",  # Crucial: include the /v1 suffix
        "api_key": "ollama",                     # Required placeholder text
    }
]

# Step 2: Create assistant agent
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant who writes and explains Python code clearly.",
    llm_config=llm_config
)

# Step 3: Create user proxy agent
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config={
        "executor": LocalCommandLineCodeExecutor(work_dir="coding", timeout=30),
    },
)

# Step 4: Start task
chat_result = user_proxy.initiate_chat(
    recipient=assistant,
    message="Plot a sine wave using matplotlib from -2π to 2π and save the plot as sine_wave.png.",
    max_turns=4,
    summary_method="reflection_with_llm"
)

# Step 5: Display image
from IPython.display import Image, display
import os

image_path = "coding/sine_wave.png"
if os.path.exists(image_path):
    display(Image(filename=image_path))
else:
    print("Plot not found.")

# Step 6: Print summary
print("\nFinal Summary:")
print(chat_result.summary)


user_proxy (to assistant):

Plot a sine wave using matplotlib from -2π to 2π and save the plot as sine_wave.png.

--------------------------------------------------------------------------------
assistant (to user_proxy):

This task requires using the `numpy` library to generate the numerical data and the `matplotlib` library to handle the plotting and saving process.

Here is the complete Python code:

```python
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the parameters (Start, End, Number of points)
start_angle = -2 * np.pi
end_angle = 2 * np.pi
num_points = 400  # Higher number means a smoother curve

# 2. Generate the x-axis data (angles from -2π to 2π)
x = np.linspace(start_angle, end_angle, num_points)

# 3. Calculate the y-axis data (y = sin(x))
y = np.sin(x)

# 4. Create the plot figure
plt.figure(figsize=(12, 6)) # Optionally set the size of the plot
plt.plot(x, y, label='sin(x)', color='blue', linewidth=2)

# 5. Add labels, title, and grid for clarity
plt.

## Human-in-the-Loop

AG2 makes integrating human feedback seamless through its human-in-the-loop functionality, allowing AI agents to collaborate with humans during workflows.  

This is crucial for:   

- **Critical decisions** requiring human judgment
- **High-stakes scenarios** with significant consequences
- **Regulatory compliance** requiring human oversight
- **Quality assurance** and validation

You can configure how and when human input is solicited using the `human_input_mode` parameter:

- ALWAYS: Requires human input for every response

- NEVER: Operates autonomously without human involvement

- TERMINATE: Only requests human input to end conversations

For convenience, AG2 provides the specialized `UserProxyAgent` class that automatically sets `human_input_mode` to ALWAYS and supports **code execution**. 

**important note**: Always use code execution functionality with caution and at your own discretion.


### Human-in-the-Loop Example: Bug Triage Bot

This example demonstrates how to use AG2’s `ConversableAgent` in `human_input_mode="ALWAYS"` to enable **human-in-the-loop workflows**.

We simulate a **bug triage assistant** (`triage_bot`) that classifies bug reports as either:
- Escalate (for example, critical crash or security issue),
- Close (for example, minor cosmetic issue),
- Medium priority (default for others).

For each classification, the assistant **asks the human agent for confirmation or correction**. This ensures the AI doesn’t act on high-impact decisions without oversight.

At the end, the assistant summarizes the triage results.

---

### Try these inputs when prompted

When you’re prompted to reply as the human agent, try responding with the following:

- **Confirm assistant’s suggestion**  
  `"Yes, escalate it."`  
  `"Closing this makes sense."`

- **Override assistant’s suggestion**  
  `"This should be marked as high priority instead."`  
  `"Let’s keep this open for now."`

- **Ask for clarification**  
  `"Why do you think this is low priority?"`  
  `"Can you provide more reasoning?"`

You can also type `exit` at any time to end the conversation.


In [ ]:
from autogen import ConversableAgent
import random

# Step 1: LLM configuration
llm_config = [
    {
        "model": "gemma4",
        "base_url": "http://localhost:11434/v1",  # Crucial: include the /v1 suffix
        "api_key": "ollama",                     # Required placeholder text
    }
]

# Step 2: Define system message
triage_system_message = """
You are a bug triage assistant. You will be given bug report summaries.

For each bug:
- If it is urgent (e.g., 'crash', 'security', or 'data loss' is mentioned), escalate it and ask the human agent for confirmation.
- If it seems minor (e.g., cosmetic, typo), suggest closing it but still ask for human review.
- Otherwise, classify it as medium priority and ask the human for review.

Once all bugs are processed, summarize what was escalated, closed, or marked as medium priority.
End by saying: "You can type exit to finish."
"""

# Step 3: Create assistant agent
triage_bot = ConversableAgent(
    name="triage_bot",
    system_message=triage_system_message,
    llm_config=llm_config
)

# Step 4: Create human agent
human = ConversableAgent(
    name="human",
    human_input_mode="ALWAYS",
)

# Step 5: Generate sample bugs
BUGS = [
    "App crashes when opening user profile.",
    "Minor UI misalignment on settings page.",
    "Password reset email not sent consistently.",
    "Typo in the About Us footer text.",
    "Database connection timeout under heavy load.",
    "Login form allows SQL injection attack.",
]

random.shuffle(BUGS)
selected_bugs = BUGS[:3]

# Step 6: Format prompt
initial_prompt = (
    "Please triage the following bug reports one by one:\n\n" +
    "\n".join([f"{i+1}. {bug}" for i, bug in enumerate(selected_bugs)])
)

# Step 7: Start conversation
human.initiate_chat(
    recipient=triage_bot,
    message=initial_prompt,
)

human (to triage_bot):

Please triage the following bug reports one by one:

1. Minor UI misalignment on settings page.
2. App crashes when opening user profile.
3. Login form allows SQL injection attack.

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
triage_bot (to human):

**Bug Report 1: Minor UI misalignment on settings page.**
*   **Priority:** Minor
*   **Recommendation:** Suggest closing this issue as low priority, but please have a human review before closure.

**Bug Report 2: App crashes when opening user profile.**
*   **Priority:** Urgent (Crash/Stability)
*   **Recommendation:** **ESCALATION.** This is a crash bug. Please confirm if this should be immediately escalated to stop the current release cycle.

**Bug Report 3: Login form allows SQL injection attack.**
*   **Priority:** Urgent (Security/Data Loss)
*   **Recommendation:** **ESCALATION.** This is a critical security vulnerability. Please confirm if this

## Agent Orchestration & Multi-Agent Systems in AG2<a name="agent-orchestration"></a>

AG2 enables the coordination of multiple intelligent agents to collaboratively solve complex tasks. This is known as **agent orchestration** — a powerful design pattern where each agent plays a specialized role, and a **group manager** handles the conversation flow.

### Why Multi-Agent Systems?

Many real-world problems require more than just a single AI assistant. With AG2, you can:

- Assign specific roles and responsibilities to different agents.
- Orchestrate conversations using built-in patterns (for example, Auto, RoundRobin, Manual).
- Enable agents to build on each other's outputs and refine ideas.
- Incorporate human agents for oversight, decisions, or approval.

### Orchestration Patterns in AG2

AG2 provides several orchestration patterns to structure agent interactions:

- **Two-Agent Chat**: Simple back-and-forth between two agents.
- **Sequential Chat**: Conversations where one agent’s output becomes another’s input.
- **Group Chat**: Multiple agents interact, with selection logic for who speaks next.
- **Nested Chat**: Reusable sub-conversations packaged as a single workflow.

---

## GroupChat and GroupChatManager

In AG2, multi-agent collaboration is coordinated using `GroupChat` and `GroupChatManager`.

### GroupChat

`GroupChat` defines a team of agents and how they interact in a shared conversation. It includes:

- **Agents**: A list of AI (or human) agents that participate in the group dialogue.
- **Speaker Selection Method**: Determines which agent speaks next. Options include:
  - `"auto"`: Uses the LLM to select the most contextually appropriate agent.
  - `"round_robin"`: Agents take turns in a fixed sequence.
  - `"manual"`: Human selects the next speaker.
  - `"random"`: Agents are chosen randomly.

This structure allows you to define collaborative workflows where agents build on each other’s contributions.

### GroupChatManager

`GroupChatManager` is responsible for managing the flow of the group conversation and it:

- Orchestrates message passing between agents.
- Decides when to stop the conversation (for example, based on a termination condition or turn limit).
- Leverages the speaker selection method defined in `GroupChat`.

It acts like a facilitator that ensures the conversation runs according to the logic you've configured.

### How It Works

1. A conversation is initiated by one of the agents (often a user or teacher agent).
2. The `GroupChatManager` uses the selected pattern (for example, AutoPattern) to determine which agent speaks next.
3. Agents take turns responding based on their roles and system messages.
4. The conversation ends either when a stop condition (for example, a message like "DONE") is met or when a maximum number of turns is reached.

This setup enables modular, role-based collaboration — ideal for use cases like lesson planning, research workflows, or multi-perspective decision-making.

To explore more orchestration patterns, visit the AG2 documentation at:  
https://docs.ag2.ai/latest/docs/user-guide/advanced-concepts/orchestration/group-chat/introduction/

---

### Example: Group Chat for Lesson Planning

This example shows how a teacher agent collaborates with a planner and a reviewer to create a lesson plan, using AG2’s `GroupChat` and `AutoPattern` to manage the conversation.

**Note:** The `is_termination_msg` parameter used for `teacher` agent defines a custom rule to end the conversation —  
in this case, the workflow stops when the teacher replies with "DONE!".


In [ ]:
from autogen import ConversableAgent, GroupChat, GroupChatManager

# LLM configuration
llm_config = [
    {
        "model": "gemma4",
        "base_url": "http://localhost:11434/v1",  # Crucial: include the /v1 suffix
        "api_key": "ollama",                     # Required placeholder text
    }
]

# System messages
planner_message = "Create a short lesson plan for 4th graders."
reviewer_message = "Review a plan and suggest up to 3 brief edits."
teacher_message = "Suggest a topic and reply DONE when satisfied."

# Create agents
lesson_planner = ConversableAgent(
    name="planner_agent",
    system_message=planner_message,
    description="Makes lesson plans.",
    llm_config=llm_config
)

lesson_reviewer = ConversableAgent(
    name="reviewer_agent",
    system_message=reviewer_message,
    description="Reviews lesson plans and suggests edits.",
    llm_config=llm_config
)

teacher = ConversableAgent(
    name="teacher_agent",
    system_message=teacher_message,
    llm_config=llm_config,
    is_termination_msg=lambda x: "DONE" in (x.get("content", "") or "").upper()
)

# Create group chat
groupchat = GroupChat(
    agents=[teacher, lesson_planner, lesson_reviewer],
    speaker_selection_method="auto"
)

# Create manager
manager = GroupChatManager(
    name="group_manager",
    groupchat=groupchat,
    llm_config=llm_config
)

# Start chat
teacher.initiate_chat(
    recipient=manager,
    message="Make a simple lesson about the moon.",
    max_turns=6,
    summary_method="reflection_with_llm"
)

teacher_agent (to group_manager):

Make a simple lesson about the moon.

--------------------------------------------------------------------------------
[autogen.oai.client: 09-26 18:13:03] {734} WARNING - Model gemma4 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.

Next speaker: planner_agent


>>>>>>>> USING AUTO REPLY...
[autogen.oai.client: 09-26 18:13:50] {734} WARNING - Model gemma4 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
planner_agent (to group_manager):

# 🌙 Lesson Plan: Exploring Our Lunar Neighbor

**Grade Level:** 4th Grade (Ages 9-10)
**Time Allotment:** 35–40 Minutes
**Topic:** The Moon: Phases, Movement, and Facts

---

## ⭐ Learning Objectives (Students will be able to):

1.  Define the Moon as Earth's natural satellite.
2.  Explain that the Moon does n

## Tools and Extensions<a name="tools"></a>

Tools extend agent capabilities beyond text conversations, allowing them to:
- Execute code
- Access external APIs
- Perform calculations
- Interact with databases
- Generate visualizations

Agents gain significant utility through tools as they provide access to external data, APIs, and functionality.  

Let's explore how to integrate tools with AG2 agents:


In [2]:
from autogen import ConversableAgent, register_function
from typing import Annotated

# Step 1: LLM configuration
llm_config = [
    {
        "model": "gemma4",
        "base_url": "http://localhost:11434/v1",  # Crucial: include the /v1 suffix
        "api_key": "ollama",                     # Required placeholder text
    }
]

# Step 2: Define function
def is_prime(n: Annotated[int, "Positive integer"]) -> str:
    if n < 2:
        return "No"
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return "No"
    return "Yes"

# Step 3: Create agents
math_asker = ConversableAgent(
    name="math_asker",
    system_message="Ask whether a number is prime.",
    llm_config=llm_config
)

math_checker = ConversableAgent(
    name="math_checker",
    human_input_mode="NEVER",
    llm_config=llm_config
)

# Step 4: Register function
register_function(
    is_prime,
    caller=math_asker,
    executor=math_checker,
    description="Check if a number is prime. Returns Yes or No."
)

# Step 5: Start conversation
math_checker.initiate_chat(
    recipient=math_asker,
    message="Is 72 a prime number?",
    max_turns=2
)

math_checker (to math_asker):

Is 72 a prime number?

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
[autogen.oai.client: 09-26 18:35:27] {734} WARNING - Model gemma4 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
math_asker (to math_checker):


***** Suggested tool call (call_qngncuzz): is_prime *****
Arguments: 
{"n":72}
*********************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION is_prime...
Call ID: call_qngncuzz
Input arguments: {'n': 72}

>>>>>>>> EXECUTED FUNCTION is_prime...
Call ID: call_qngncuzz
Input arguments: {'n': 72}
Output:
No
math_checker (to math_asker):

***** Response from calling tool (call_qngncuzz) *****
No
******************************************************

---------------------------

ChatResult(chat_id=295403219295574036062296114532247437473, chat_history=[{'content': 'Is 72 a prime number?', 'role': 'assistant', 'name': 'math_checker'}, {'content': '', 'tool_calls': [{'id': 'call_qngncuzz', 'function': {'arguments': '{"n":72}', 'name': 'is_prime'}, 'type': 'function', 'index': 0}], 'role': 'assistant'}, {'content': 'No', 'tool_responses': [{'tool_call_id': 'call_qngncuzz', 'role': 'tool', 'content': 'No'}], 'role': 'tool', 'name': 'math_checker'}, {'content': 'No, 72 is not a prime number.', 'role': 'user', 'name': 'math_asker'}], summary='No, 72 is not a prime number.', cost={'usage_including_cached_inference': {'total_cost': 0, 'gemma4': {'cost': 0, 'prompt_tokens': 204, 'completion_tokens': 85, 'total_tokens': 289}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemma4': {'cost': 0, 'prompt_tokens': 204, 'completion_tokens': 85, 'total_tokens': 289}}}, human_input=[])

### What is `register_function(...)` ?

In AG2, `register_function(...)` is used to expose a Python function as a **tool** that can be executed by one agent on behalf of another. This enables agents to delegate tasks like computation, data processing, or external API calls.

#### Purpose
- Extend agent capabilities beyond text generation.
- Allow agents to solve tasks through **function execution**.
- Enable collaborative workflows between a **caller** and an **executor** agent.

#### Parameters
- **function**: A regular Python function to be used as a tool.
- **caller**: The agent that will request the tool to be used.
- **executor**: The agent that will actually execute the function.
- **description** *(optional)*: A natural language description of the function for the LLM to decide when to use it.

#### Example
```python
register_function(
    is_prime,
    caller=math_asker,
    executor=math_checker,
    description="Check if a number is prime. Returns Yes or No."
)


## Structured Outputs<a name="structured-outputs"></a>

Structured outputs ensure consistent, validated agent responses using Pydantic models.   

This is crucial for:
  
- **Data validation**: Ensuring response format consistency
- **API integration**: Reliable data exchange
- **Quality assurance**: Preventing malformed outputs
- **Type safety**: Clear data contracts

**Analogy:** Like standardized hospital forms ensure doctors always record patient info the same way, structured outputs make sure agents respond in predictable, machine-readable formats.


In AG2, structured outputs are implemented using Pydantic models and the `response_format` parameter in the `LLMConfig`.

To ensure that your agent always returns outputs in a consistent structure, you define a Pydantic class (for example, `ResponseModel`) and assign it to the `response_format` argument of your configuration.

This tells the underlying LLM to return a JSON-compatible response matching the defined schema.

```python
class ResponseModel(BaseModel):
    name: str
    status: str

llm_config = LLMConfig(
    api_type="openai",
    model="gpt-4o-mini",
    response_format=ResponseModel
)
```

With this setup:

- The agent automatically formats its responses to match the ResponseModel. 
- You don’t need to prompt the LLM to format its response.
- The response is parsed and validated by AG2.

This approach is essential for reliable automation, integrations, and downstream processing.

Let's implement structured outputs with AG2:


In [4]:
from pydantic import BaseModel
from autogen import ConversableAgent

# Define structured output model
class TicketSummary(BaseModel):
    customer_name: str
    issue_type: str
    urgency_level: str
    recommended_action: str

# LLM configuration
llm_config = [
    {
        "model": "",
    }
]

llm_config = {
    "config_list": [
        {
            "model": "gemma4",
            "base_url": "http://localhost:11434/v1",  # Crucial: include the /v1 suffix
            "api_key": "ollama",                     # Required placeholder text
        }
    ],
    "response_format": TicketSummary
}

# Create agent
support_agent = ConversableAgent(
    name="support_agent",
    system_message=(
        "You are a support assistant. Summarize a customer ticket using:"
        "\n- customer_name"
        "\n- issue_type (e.g. login issue, billing problem, bug report)"
        "\n- urgency_level (Low, Medium, High)"
        "\n- recommended_action"
    ),
    llm_config=llm_config
)

# Start conversation
support_agent.initiate_chat(
    recipient=support_agent,
    message="Ticket: John Doe is unable to reset his password and has an important meeting in 30 minutes.",
    max_turns=1
)

support_agent (to support_agent):

Ticket: John Doe is unable to reset his password and has an important meeting in 30 minutes.

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
[autogen.oai.client: 09-26 19:08:48] {734} WARNING - Model gemma4 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
support_agent (to support_agent):

{
  "customer_name": "John Doe",
  "issue_type": "Login issue",
  "urgency_level": "High",
  "recommended_action": "Manually reset the user's password or engage Security/Tier 2 support for immediate account access assistance."
}

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (c0fb676f-a789-4b94-ba26-fd30a2f99207): Maximum turns (1) reached


ChatResult(chat_id=207894040912328737766243273990411736430, chat_history=[{'content': 'Ticket: John Doe is unable to reset his password and has an important meeting in 30 minutes.', 'role': 'assistant', 'name': 'support_agent'}, {'content': 'Ticket: John Doe is unable to reset his password and has an important meeting in 30 minutes.', 'role': 'user', 'name': 'support_agent'}, {'content': '{\n  "customer_name": "John Doe",\n  "issue_type": "Login issue",\n  "urgency_level": "High",\n  "recommended_action": "Manually reset the user\'s password or engage Security/Tier 2 support for immediate account access assistance."\n}', 'role': 'assistant', 'name': 'support_agent'}, {'content': '{\n  "customer_name": "John Doe",\n  "issue_type": "Login issue",\n  "urgency_level": "High",\n  "recommended_action": "Manually reset the user\'s password or engage Security/Tier 2 support for immediate account access assistance."\n}', 'role': 'user', 'name': 'support_agent'}], summary='{\n  "customer_name": 

## Conclusion<a name="conclusion"></a>

Congratulations! You've completed a comprehensive tour of AG2's basic concepts. Let's recap what we've learned:

### Key concepts mastered:

- **LLM Configuration** - The foundation that connects agents to language models
- **ConversableAgent** - The core building block for all AG2 agents
- **Human in the Loop (HITL)** - Enabling human oversight in AI workflows
- **Agent Orchestration** - Coordinating multiple agents for complex tasks
- **Tools & Extensions** - Extending agent capabilities beyond text
- **Structured Outputs** - Ensuring consistent, validated responses

### Practical skills developed:

- Creating and configuring specialized agents
- Implementing multi-agent collaboration patterns
- Building HITL workflows for critical decisions
- Integrating tools and external capabilities
- Validating outputs with structured schemas
- Following security and operational best practices

### Next steps:

Now that you understand AG2's basic concepts, you can:

1. **Build Domain-Specific Applications** - Apply these concepts to your specific use case.
2. **Explore Advanced Features** - Dive into code execution, web scraping, and API integrations.
3. **Scale Your Systems** - Learn about production deployment and monitoring.
4. **Join the Community** - Contribute to the AG2 ecosystem and learn from others.

### Things to remember:

- **Start simple** and gradually add complexity.
- **Test thoroughly** with various scenarios.
- **Monitor performance** and adjust configurations.
- **Follow best practices** for security and reliability.
- **Keep learning** as AG2 continues to evolve.

AG2 provides a powerful foundation for building intelligent, collaborative AI systems. The concepts you've learned here will serve as building blocks for more advanced implementations.  

### Advanced agentic design patterns:  
  
AG2 supports advanced features like custom rules for conversation endings, RAG, code execution, and secure tool use. You can learn more from the official documentation.  
  
Happy building with AG2! 🤖✨
